# EM Algorithm for Factor Analysis

## Learning Objectives
- Derive the **E-step**: posterior $z^{(i)} \mid x^{(i)} \sim \mathcal{N}(\mu_{z|x},\, \Sigma_{z|x})$ via the conditional Gaussian formula
- Implement the **M-step** updates for $\mu$, $\Lambda$, and $\Psi$ using posterior moments $E[z]$ and $E[zz^T]$
- Understand why the M-step uses $E[zz^T] = \mu_{z|x}\mu_{z|x}^T + \Sigma_{z|x}$ — not just $E[z]E[z]^T$
- Verify that the EM log-likelihood increases monotonically
- Compare Factor Analysis with PCA as a dimensionality reduction method

## Problem Statement

**Model:** $z \sim \mathcal{N}(0, I)$, $x|z \sim \mathcal{N}(\mu + \Lambda z, \Psi)$.

Marginal $\ell(\mu, \Lambda, \Psi)$ has no closed-form maximiser — EM is required.

---

**E-step** — compute posterior of $z^{(i)}$ given $x^{(i)}$ using the conditional Gaussian formula:
$$z^{(i)} \mid x^{(i)} \sim \mathcal{N}(\mu_{z|x}^{(i)},\; \Sigma_{z|x})$$
$$\mu_{z|x}^{(i)} = \Lambda^T(\Lambda\Lambda^T + \Psi)^{-1}(x^{(i)} - \mu)$$
$$\Sigma_{z|x} = I - \Lambda^T(\Lambda\Lambda^T + \Psi)^{-1}\Lambda$$

Note: $\Sigma_{z|x}$ is the same for every data point.

---

**M-step** — maximise $\sum_i E_{z^{(i)}|x^{(i)}}[\log p(x^{(i)}, z^{(i)}; \mu, \Lambda, \Psi)]$ over the parameters.

The needed moments are:
$$E[z^{(i)}] = \mu_{z|x}^{(i)}, \qquad
E[z^{(i)}{z^{(i)}}^T] = \mu_{z|x}^{(i)}{\mu_{z|x}^{(i)}}^T + \Sigma_{z|x}$$

**Updates:**
$$\mu \leftarrow \frac{1}{m}\sum_i x^{(i)}$$
$$\Lambda \leftarrow \left(\sum_i (x^{(i)}-\mu)E[z^{(i)}]^T\right)\left(\sum_i E[z^{(i)}{z^{(i)}}^T]\right)^{-1}$$
$$\Psi_{jj} \leftarrow \frac{1}{m}\sum_i \left[x^{(i)}{x^{(i)}}^T - x^{(i)}E[z^{(i)}]^T\Lambda^T - \Lambda E[z^{(i)}]{x^{(i)}}^T + \Lambda E[z^{(i)}{z^{(i)}}^T]\Lambda^T\right]_{jj}$$

## 1. EM for Factor Analysis: Full Implementation

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal

def fa_em(X, k, max_iters=100, tol=1e-6, seed=None):
    """
    EM algorithm for Factor Analysis.
    Model: z ~ N(0,I), x|z ~ N(mu + Lambda*z, Psi)
    """
    rng = np.random.default_rng(seed)
    m, n = X.shape

    # Initialise
    mu     = X.mean(axis=0)
    Lambda = rng.standard_normal((n, k)) * 0.1
    Psi    = np.var(X, axis=0)   # diagonal noise

    ll_hist = []

    for iteration in range(max_iters):
        # --- E-step ---
        # Marginal covariance: Sigma_x = Lambda Lambda^T + Psi_mat
        Psi_mat  = np.diag(Psi)
        Sig_x    = Lambda @ Lambda.T + Psi_mat
        Sig_x_inv = np.linalg.inv(Sig_x)

        M = Lambda.T @ Sig_x_inv   # (k, n)  — used in posterior mean

        # Posterior covariance: Sigma_{z|x} = I - M Lambda  (same for all i)
        Sig_zx = np.eye(k) - M @ Lambda

        # Posterior means: mu_{z|x}^(i) = M (x^(i) - mu)
        Xc = X - mu                  # (m, n)
        Ez = Xc @ M.T                # (m, k)  E[z^(i)]

        # E[z z^T] = mu_{z|x} mu_{z|x}^T + Sigma_{z|x}  (summed over i)
        Ezz = Ez.T @ Ez + m * Sig_zx  # (k, k)

        # Log-likelihood (marginal)
        try:
            ll = 0.0
            sign, logdet = np.linalg.slogdet(Sig_x)
            ll = -0.5 * m * (n * np.log(2 * np.pi) + logdet) \
                 - 0.5 * np.sum(Xc * (Xc @ Sig_x_inv))
        except Exception:
            ll = np.nan
        ll_hist.append(ll)

        # --- M-step ---
        # Update mu
        mu = X.mean(axis=0)

        # Update Lambda: (sum_i (x-mu) E[z]^T) @ inv(sum_i E[zz^T])
        Lambda_new = (Xc.T @ Ez) @ np.linalg.inv(Ezz)   # (n, k)

        # Update Psi (diagonal only)
        Phi = (Xc.T @ Xc
               - Lambda_new @ Ez.T @ Xc
               - Xc.T @ Ez @ Lambda_new.T
               + Lambda_new @ Ezz @ Lambda_new.T) / m
        Psi_new = np.diag(Phi)   # keep only diagonal
        Psi_new = np.maximum(Psi_new, 1e-6)  # numerical floor

        Lambda = Lambda_new
        Psi    = Psi_new

        if len(ll_hist) >= 2 and abs(ll_hist[-1] - ll_hist[-2]) < tol:
            break

    return mu, Lambda, Psi, ll_hist


# Generate data from a known Factor Analysis model
np.random.seed(42)
n_true, k_true, m_data = 6, 2, 100
mu_true     = np.array([1.0, -0.5, 0.5, 2.0, -1.0, 0.0])
Lambda_true = np.array([[1.5, 0.2],
                         [0.3, 1.2],
                         [1.0, -0.5],
                         [0.1, 1.4],
                         [-0.8, 0.9],
                         [1.1, 0.3]])
Psi_true    = np.array([0.3, 0.5, 0.2, 0.4, 0.6, 0.3])

z_data   = np.random.randn(m_data, k_true)
eps_data = np.array([np.random.multivariate_normal(np.zeros(n_true), np.diag(Psi_true))
                     for _ in range(m_data)])
X_data   = mu_true + z_data @ Lambda_true.T + eps_data

mu_hat, Lambda_hat, Psi_hat, ll_hist = fa_em(X_data, k=k_true, max_iters=200, seed=0)

print(f'Converged in {len(ll_hist)} iterations')
print(f'Final log-likelihood: {ll_hist[-1]:.2f}')
print(f'\nTrue μ:  {mu_true.round(3)}')
print(f'Est  μ:  {mu_hat.round(3)}')
print(f'\nTrue Ψ (diagonal):  {Psi_true.round(3)}')
print(f'Est  Ψ (diagonal):  {Psi_hat.round(3)}')

## 2. Log-Likelihood Convergence and E-step Posterior

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: log-likelihood curve
ax = axes[0]
ax.plot(range(1, len(ll_hist)+1), ll_hist, 'b-o', lw=2.5, ms=5)
ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log-likelihood $\\ell$')
ax.set_title('FA-EM: log-likelihood increases monotonically')

# Middle: delta log-likelihood (shows convergence speed)
ax = axes[1]
deltas = np.diff(ll_hist)
ax.semilogy(range(1, len(deltas)+1), deltas + 1e-10, 'r-o', lw=2.5, ms=5)
ax.set_xlabel('Iteration')
ax.set_ylabel('$\\Delta\\ell$ (log scale)')
ax.set_title('Improvement per iteration\n(exponential convergence)')

# Right: posterior means (E-step output) for 2 factors
ax = axes[2]
Psi_mat   = np.diag(Psi_hat)
Sig_x     = Lambda_hat @ Lambda_hat.T + Psi_mat
M         = Lambda_hat.T @ np.linalg.inv(Sig_x)
Xc        = X_data - mu_hat
Ez_final  = Xc @ M.T   # (m, k) posterior means
Sig_zx    = np.eye(k_true) - M @ Lambda_hat

ax.scatter(Ez_final[:, 0], Ez_final[:, 1], s=25, alpha=0.6, c='steelblue')
# Draw 1-sigma posterior uncertainty ellipse (same for all points)
from matplotlib.patches import Ellipse
vals, vecs = np.linalg.eigh(Sig_zx)
angle = np.degrees(np.arctan2(*vecs[:, -1][::-1]))
ell = Ellipse(xy=(0, 0), width=2*np.sqrt(vals[-1]), height=2*np.sqrt(vals[0]),
              angle=angle, fc='none', ec='red', lw=2, ls='--')
ax.add_patch(ell)
ax.set_xlabel('$E[z_1 | x]$'); ax.set_ylabel('$E[z_2 | x]$')
ax.set_title('E-step posterior means $\\mu_{z|x}^{(i)}$\nRed ellipse = posterior uncertainty $\\Sigma_{z|x}$')
ax.text(0.03, 0.97, f'$\\Sigma_{{z|x}}$ trace = {np.trace(Sig_zx):.3f}\n(same for all points)',
        transform=ax.transAxes, fontsize=8.5, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('FA-EM Convergence and E-step Posterior', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Why $E[zz^T]$ ≠ $E[z]E[z]^T$: The Posterior Covariance Matters

The M-step $\Lambda$ update uses:
$$E[z^{(i)}{z^{(i)}}^T] = \underbrace{\mu_{z|x}^{(i)}{\mu_{z|x}^{(i)}}^T}_{\text{point estimate}} + \underbrace{\Sigma_{z|x}}_{\text{posterior uncertainty}}$$

Ignoring $\Sigma_{z|x}$ (using only $E[z]E[z]^T$) produces a **biased** update — a common mistake.
The correction term $\Sigma_{z|x}$ is exactly what ensures EM increases $\ell$ at each step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compare correct EM vs biased EM (ignoring Sigma_{z|x})

def fa_em_biased(X, k, max_iters=100, tol=1e-6, seed=None):
    """EM with the bug: uses E[z]E[z]^T instead of E[zz^T]."""
    rng = np.random.default_rng(seed)
    m, n = X.shape
    mu     = X.mean(axis=0)
    Lambda = rng.standard_normal((n, k)) * 0.1
    Psi    = np.var(X, axis=0)
    ll_hist = []

    for _ in range(max_iters):
        Psi_mat  = np.diag(Psi)
        Sig_x    = Lambda @ Lambda.T + Psi_mat
        Sig_x_inv = np.linalg.inv(Sig_x)
        M = Lambda.T @ Sig_x_inv
        Xc = X - mu
        Ez = Xc @ M.T   # (m, k)

        # BUG: ignore Sigma_{z|x}
        Ezz_biased = Ez.T @ Ez    # missing + m * Sig_zx

        sign, logdet = np.linalg.slogdet(Sig_x)
        ll = -0.5 * m * (n * np.log(2 * np.pi) + logdet) \
             - 0.5 * np.sum(Xc * (Xc @ Sig_x_inv))
        ll_hist.append(ll)

        mu = X.mean(axis=0)
        Lambda_new = (Xc.T @ Ez) @ np.linalg.inv(Ezz_biased)
        Phi = (Xc.T @ Xc - Lambda_new @ Ez.T @ Xc
               - Xc.T @ Ez @ Lambda_new.T
               + Lambda_new @ Ezz_biased @ Lambda_new.T) / m
        Psi_new = np.maximum(np.diag(Phi), 1e-6)
        Lambda, Psi = Lambda_new, Psi_new

        if len(ll_hist) >= 2 and abs(ll_hist[-1] - ll_hist[-2]) < tol:
            break
    return ll_hist

ll_correct = ll_hist   # already computed above
ll_biased  = fa_em_biased(X_data, k=k_true, max_iters=200, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(range(1, len(ll_correct)+1), ll_correct, 'b-', lw=2.5, label='Correct EM')
ax.plot(range(1, len(ll_biased)+1),  ll_biased,  'r--', lw=2.5, label='Biased EM (ignores $\\Sigma_{z|x}$)')
ax.set_xlabel('Iteration'); ax.set_ylabel('Log-likelihood')
ax.set_title('Correct vs. biased EM\n(monotonicity violated in biased version)')
ax.legend(fontsize=9)

# Right: visualise Sigma_{z|x} contribution
ax = axes[1]
Sig_zx_vals = np.diag(Sig_zx)   # diagonal of posterior covariance
ax.bar(range(k_true), Sig_zx_vals, color='#2166ac', edgecolor='k', alpha=0.8)
ax.set_xticks(range(k_true))
ax.set_xticklabels([f'Factor {j+1}' for j in range(k_true)])
ax.set_ylabel('Posterior variance $\\Sigma_{z_j|x}$')
ax.set_title('E-step posterior uncertainty\n'
             'M-step adds this to $E[z]E[z]^T$ to get $E[zz^T]$')
ax.text(0.5, 0.95, '$E[zz^T] = E[z]E[z]^T + \\Sigma_{z|x}$',
        transform=ax.transAxes, ha='center', va='top', fontsize=10,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('The Critical Role of Posterior Covariance in the M-step',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Factor Analysis vs. PCA for Dimensionality Reduction

Both FA and PCA find a low-dimensional subspace. Key differences:

| | Factor Analysis | PCA |
|---|---|---|
| Noise model | Heteroscedastic diagonal $\Psi$ | Implicit (equal variance) |
| Objective | Maximise marginal likelihood | Maximise variance explained |
| Latent posterior | $p(z|x)$ fully characterised | Point projection |
| Handles $n \gg m$? | Yes | Yes |
| Rotation invariance | $\Lambda$ not identifiable (need ICA) | Eigenvectors unique |
| Probabilistic? | Yes — generative model | No (unless Probabilistic PCA) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FactorAnalysis, PCA

np.random.seed(1)

# Data: 4D observed with heteroscedastic noise (FA should outperform PCA here)
n_d, k_d, m_d = 4, 1, 80
Lambda_d = np.array([[2.0], [1.5], [0.5], [0.3]])
# Heteroscedastic noise: last 2 features very noisy
Psi_d = np.array([0.1, 0.1, 2.0, 2.0])

z_d   = np.random.randn(m_d, k_d)
eps_d = np.array([np.random.multivariate_normal(np.zeros(n_d), np.diag(Psi_d))
                  for _ in range(m_d)])
X_d   = z_d @ Lambda_d.T + eps_d

# Fit FA and PCA
fa_sk = FactorAnalysis(n_components=k_d, random_state=0)
Z_fa  = fa_sk.fit_transform(X_d)

pca = PCA(n_components=k_d)
Z_pca = pca.fit_transform(X_d)

# Correlation of recovered factor with true z
corr_fa  = abs(np.corrcoef(Z_fa[:, 0],  z_d[:, 0])[0, 1])
corr_pca = abs(np.corrcoef(Z_pca[:, 0], z_d[:, 0])[0, 1])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: FA recovered factor vs true z
ax = axes[0]
ax.scatter(z_d[:, 0], Z_fa[:, 0], s=40, alpha=0.7, c='#2166ac')
ax.set_xlabel('True $z$'); ax.set_ylabel('FA recovered factor')
ax.set_title(f'Factor Analysis\nCorr with true z = {corr_fa:.3f}')

# Middle: PCA recovered factor vs true z
ax = axes[1]
ax.scatter(z_d[:, 0], Z_pca[:, 0], s=40, alpha=0.7, c='#d6604d')
ax.set_xlabel('True $z$'); ax.set_ylabel('PCA first component')
ax.set_title(f'PCA\nCorr with true z = {corr_pca:.3f}')

# Right: loadings comparison
ax = axes[2]
feat_idx = np.arange(n_d)
w = 0.3
lambda_true_norm = Lambda_d[:, 0] / np.linalg.norm(Lambda_d[:, 0])
lambda_fa_norm   = fa_sk.components_[0] / np.linalg.norm(fa_sk.components_[0])
lambda_pca_norm  = pca.components_[0] / np.linalg.norm(pca.components_[0])

# Align signs
if np.dot(lambda_fa_norm,  lambda_true_norm) < 0: lambda_fa_norm  = -lambda_fa_norm
if np.dot(lambda_pca_norm, lambda_true_norm) < 0: lambda_pca_norm = -lambda_pca_norm

ax.bar(feat_idx - w, lambda_true_norm, width=w, label='True Λ', color='gray',  edgecolor='k', alpha=0.8)
ax.bar(feat_idx,     lambda_fa_norm,   width=w, label='FA est', color='#2166ac', edgecolor='k', alpha=0.8)
ax.bar(feat_idx + w, lambda_pca_norm,  width=w, label='PCA',    color='#d6604d', edgecolor='k', alpha=0.8)
ax.set_xticks(feat_idx)
ax.set_xticklabels([f'$x_{j+1}$' for j in range(n_d)])
ax.set_ylabel('Loading (normalised)')
ax.set_title('FA vs PCA loadings vs truth\n(FA correctly down-weights noisy features)')
ax.legend(fontsize=9)

# Annotation: PCA is pulled toward noisy features
ax.text(0.5, 0.03, f'Noisy features: $\\Psi_3={Psi_d[2]}, \\Psi_4={Psi_d[3]}$\n'
        'FA weights by SNR; PCA does not',
        transform=ax.transAxes, ha='center', fontsize=8.5,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('FA vs PCA: Factor Analysis Accounts for Heteroscedastic Noise',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'FA correlation with true z:  {corr_fa:.4f}')
print(f'PCA correlation with true z: {corr_pca:.4f}')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">FA parameters</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">&#x3bc;, &#x39b;, &#x3a8;</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >initialise; want to maximise marginal log-likelihood of x</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >E-step: compute p(z|x; &#x3bc;,&#x39b;,&#x3a8;) analytically</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">E-step</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">E[z|x] and</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >E[z|x] = &#x39b;&#x1d40;(&#x39b;&#x39b;&#x1d40;+&#x3a8;)&#x207b;&#xb9;(x-&#x3bc;); E[zz&#x1d40;|x] = Var(z|x) + E[z|x]E[z|x]&#x1d40;</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >M-step: weighted MLE using these expectations</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">M-step</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">update &#x39b;, &#x3a8;</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >&#x39b; = (&#x2211;&#x2071; x&#x2071;E[z&#x2071;]&#x1d40;)(&#x2211;&#x2071; E[z&#x2071;z&#x2071;&#x1d40;])&#x207b;&#xb9;; &#x3a8; diagonal by design</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >&#x2113; increases &#x2014; repeat until convergence</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Converged</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">FA model</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >k latent factors explain low-rank structure in the data covariance</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| E-step posterior mean | $E[z^{(i)}\lvert x^{(i)}] = \Lambda^T(\Lambda\Lambda^T+\Psi)^{-1}(x^{(i)}-\mu)$ | Uses Gaussian conditional formula |
| E-step posterior cov | $\text{Var}(z^{(i)}\lvert x^{(i)}) = I - \Lambda^T(\Lambda\Lambda^T+\Psi)^{-1}\Lambda$ | Must include this in $E[z^{(i)}{z^{(i)}}^T\lvert x^{(i)}]$ |
| M-step $\Lambda$ | $\Lambda = \left(\sum_i x^{(i)}E[z^{(i)}]^T\right)\left(\sum_i E[z^{(i)}{z^{(i)}}^T]\right)^{-1}$ | Regression of $x$ on latent factor expectations |
| M-step $\Psi$ | Diagonal of $\frac{1}{m}\sum_i x^{(i)}{x^{(i)}}^T - \Lambda\frac{1}{m}\sum_i E[z^{(i)}]{x^{(i)}}^T$ | Residual variance after factor removal |
| Key subtlety | $E[z^{(i)}{z^{(i)}}^T] \neq E[z^{(i)}]E[z^{(i)}]^T$ | Posterior covariance must be included |

**Key insight:** the EM algorithm for factor analysis uses the Gaussian conditional formulas to compute the E-step in closed form; the critical subtlety is that $E[zz^T] = \text{Var}(z\lvert x) + E[z\lvert x]E[z\lvert x]^T$ — ignoring the posterior variance causes bias in the M-step.